In [ ]:
# Cell 1: Install required libraries and import everything

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score, accuracy_score
import time
import gc
import random
from copy import deepcopy
import json
import subprocess
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger("transformers").setLevel(logging.ERROR)


# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")


In [ ]:
# Cell 2: Setting Standards

BATCH_SIZE = 32
MAX_LENGTH = 128
TRAIN_SIZE = 1000
VAL_RATIO = 0.1
TEST_SIZE = 1000

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

In [ ]:
# Cell 3: data preprocessing helper functions

def remove_duplicates(data):
    """Remove duplicate texts from dataset"""
    seen = set()
    unique_idx = []
    for i, ex in enumerate(data):
        if ex["text"] not in seen:
            seen.add(ex["text"])
            unique_idx.append(i)
    print(f"  Removed {len(data) - len(unique_idx)} duplicates")
    return data.select(unique_idx)


def balance_dataset(data, n_samples):
    """Take n_samples total with equal positive/negative"""
    n_per_class = n_samples // 2
    pos = [i for i, x in enumerate(data) if x["label"] == 1]
    neg = [i for i, x in enumerate(data) if x["label"] == 0]
    random.shuffle(pos)
    random.shuffle(neg)
    selected = pos[:n_per_class] + neg[:n_per_class]
    random.shuffle(selected)
    return data.select(selected)


def train_val_split(data, val_ratio=0.1):
    """Split balanced dataset into train/val maintaining class balance"""
    pos_idx = [i for i, x in enumerate(data) if x["label"] == 1]
    neg_idx = [i for i, x in enumerate(data) if x["label"] == 0]

    n_val_pos = int(len(pos_idx) * val_ratio)
    n_val_neg = int(len(neg_idx) * val_ratio)

    train_idx = pos_idx[n_val_pos:] + neg_idx[n_val_neg:]
    val_idx = pos_idx[:n_val_pos] + neg_idx[:n_val_neg]

    random.shuffle(train_idx)
    random.shuffle(val_idx)

    return data.select(train_idx), data.select(val_idx)

In [ ]:
# Cell 4: Load and prepare IMDB dataset

print("Loading IMDB dataset...")
dataset = load_dataset("stanfordnlp/imdb")
print("IMDB dataset loaded successfully!")

# Clean and balance train
print("\nProcessing training data...")
train_clean = remove_duplicates(dataset["train"])
train_balanced = balance_dataset(train_clean, TRAIN_SIZE)

# Split train/val
train_dataset, val_dataset = train_val_split(train_balanced, val_ratio=VAL_RATIO)

# Clean and balance test (also remove overlap with train/val)
print("\nProcessing test data...")
test_clean = remove_duplicates(dataset["test"])
train_val_texts = set(train_dataset["text"]) | set(val_dataset["text"])
test_no_overlap = test_clean.filter(lambda x: x["text"] not in train_val_texts)
print(f"  Removed {len(test_clean) - len(test_no_overlap)} samples overlapping with train/val")
test_dataset = balance_dataset(test_no_overlap, TEST_SIZE)

# Verification
def verify_splits(train, val, test):
    print(f"\n{'='*50}")
    print(f"Train: {len(train)} (pos: {sum(1 for x in train if x['label']==1)})")
    print(f"Val:   {len(val)}   (pos: {sum(1 for x in val if x['label']==1)})")
    print(f"Test:  {len(test)}  (pos: {sum(1 for x in test if x['label']==1)})")

    overlaps = [
        ("Train/Val", set(train["text"]) & set(val["text"])),
        ("Train/Test", set(train["text"]) & set(test["text"])),
        ("Val/Test", set(val["text"]) & set(test["text"])),
    ]
    for name, overlap in overlaps:
        status = "✓" if len(overlap) == 0 else "✗"
        print(f"{status} {name} overlap: {len(overlap)}")
    print(f"{'='*50}")

verify_splits(train_dataset, val_dataset, test_dataset)

In [ ]:
# Cell 5: Tokenize all three datasets
print("Loading tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors=None  # Don't return tensors yet
    )

# Tokenize datasets
print("Tokenizing train dataset...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing validation dataset...")
val_dataset = val_dataset.map(tokenize_function, batched=True)
print("Tokenizing test dataset...")
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
columns = ["input_ids", "attention_mask", "label"]
train_dataset.set_format(type="torch", columns=columns)
val_dataset.set_format(type="torch", columns=columns)
test_dataset.set_format(type="torch", columns=columns)

print("\nTokenization complete!")

In [ ]:
# Cell 6:
import torch
import torch.onnx

# Mock the missing attribute if it doesn't exist\n",
if not hasattr(torch.onnx, '_CAFFE2_ATEN_FALLBACK'):
    torch.onnx._CAFFE2_ATEN_FALLBACK = False

import torchvision.io
import sys

# Monkey patch torchvision.io to add VideoReader\n",
if not hasattr(torchvision.io, 'VideoReader'):
    class VideoReaderStub:
        def __init__(self, *args, **kwargs):
            raise ImportError(
                "VideoReader is not available in this torchvision version."
            )
    torchvision.io.VideoReader = VideoReaderStub

print("✓ Applied torchvision VideoReader patch")
from datasets import Dataset
ds = Dataset.from_dict({"value": [1, 2, 3]})
ds.set_format("torch")
print("✓ Dataset formatting works:", ds[0])

In [ ]:
# Cell 7: Create DataLoaders and helper functions
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm


# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0

def reset_gpu_memory_stats():
    """Reset GPU memory stats"""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def evaluate_model(model, data_loader, desc="Evaluating"):
    """Evaluate model and return metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    inference_times = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc=desc, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            start_time = time.time()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            inference_times.append(time.time() - start_time)

            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(data_loader)
    avg_inference_time = np.mean(inference_times)

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "f1": f1,
        "inference_time_per_batch": avg_inference_time
    }

In [ ]:
# Cell 8: Training function
def train_model(hyperparams, train_loader, val_loader, test_loader, epochs, lr, weight_decay,
                warmup_ratio, dropout_rate, verbose=True):
    """Train DistilBERT and return test_f1, total_trial_time, and val_f1"""

    trial_start = time.time()  # For total runtime metric

    # Reset GPU memory
    reset_gpu_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    # Load model
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        dropout=dropout_rate,
        attention_dropout=dropout_rate
    ).to(device)

    # Optimizer
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    # Training loop
    training_start = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False) if verbose else train_loader

        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            outputs.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

    training_time = time.time() - training_start

    # Evaluate
    val_metrics = evaluate_model(model, val_loader, desc="Validating")
    test_metrics = evaluate_model(model, test_loader, desc="Testing")

    total_trial_time = time.time() - trial_start

    # Clean up
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "val_f1": val_metrics["f1"],
        "test_f1": test_metrics["f1"],        # Metric 1
        "total_trial_time": total_trial_time  # Metric 2
        # Metric 3 (trials_to_best) will be tracked in your optimization loop
    }

print("Training function defined!")

In [ ]:
# Cell 9: Train baseline model with default hyperparameters
print("Training baseline model with default hyperparameters...")
print(f"{'='*60}")

# Default hyperparameters
baseline_hyperparams = {
    "learning_rate": 2e-5,
    "epochs": 3,  # Reduced from 5 to be reasonable (DistilBERT is fast but still)
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "dropout_rate": 0.1
}

print("Hyperparameters:")
for k, v in baseline_hyperparams.items():
    print(f"  {k}: {v}")

# Train baseline model
baseline_results = train_model(
    hyperparams=baseline_hyperparams,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,  # ADD THIS - required now
    epochs=baseline_hyperparams["epochs"],
    lr=baseline_hyperparams["learning_rate"],
    weight_decay=baseline_hyperparams["weight_decay"],
    warmup_ratio=baseline_hyperparams["warmup_ratio"],
    dropout_rate=baseline_hyperparams["dropout_rate"],
    verbose=True
)

print(f"\n{'='*60}")
print("BASELINE RESULTS:")
print(f"{'='*60}")
print(f"Validation F1:       {baseline_results['val_f1']:.4f}")
print(f"Test F1:             {baseline_results['test_f1']:.4f}")  # Your main metric
print(f"Total Trial Time:    {baseline_results['total_trial_time']:.2f} seconds")  # Runtime metric
print(f"{'='*60}")

# Store baseline for later comparison
baseline_f1 = baseline_results['test_f1']
baseline_time = baseline_results['total_trial_time']

In [ ]:
# Cell 10: Define unified hyperparameter search space for all algorithms
print("=" * 60)
print("HYPERPARAMETER SEARCH SPACE")
print("=" * 60)

# Search space definition
# Format: [lower_bound, upper_bound, scale_type]
search_space = {
    "learning_rate": {
        "low": 1e-6,
        "high": 5e-5,
        "scale": "log",
        "description": "Learning rate for AdamW optimizer"
    },
    "epochs": {
        "low": 2,
        "high": 4,
        "scale": "int",
        "description": "Number of training epochs"
    },
    "weight_decay": {
        "low": 1e-5,
        "high": 1e-1,
        "scale": "log",
        "description": "L2 regularization strength"
    },
    "warmup_ratio": {
        "low": 0.0,
        "high": 0.2,
        "scale": "linear",
        "description": "Fraction of steps for LR warmup"
    },
    "dropout_rate": {
        "low": 0.05,
        "high": 0.4,
        "scale": "linear",
        "description": "Dropout probability for attention and hidden layers"
    }
}

# Print search space
for param, bounds in search_space.items():
    print(f"\n{param}:")
    print(f"  Range: [{bounds['low']}, {bounds['high']}]")
    print(f"  Scale: {bounds['scale']}")
    print(f"  {bounds['description']}")

# Extract bounds for optimization algorithms
param_names = list(search_space.keys())
lb = [search_space[p]["low"] for p in param_names]   # Lower bounds
ub = [search_space[p]["high"] for p in param_names]   # Upper bounds
scales = [search_space[p]["scale"] for p in param_names]

print(f"\n{'='*60}")
print("Bounds arrays for algorithms:")
print(f"  Parameters: {param_names}")
print(f"  Lower bounds: {lb}")
print(f"  Upper bounds: {ub}")
print(f"  Scales: {scales}")
print(f"{'='*60}")

# Number of evaluations per algorithm
N_EVALUATIONS = 20
print(f"\n⚠ Each algorithm will run {N_EVALUATIONS} evaluations")
print(f"   Total across 3 algorithms: {N_EVALUATIONS * 3} training runs")

In [ ]:
# Cell 11: Lightweight training function for hyperparameter optimization
def train_model_fast(epochs, lr, weight_decay, warmup_ratio, dropout_rate):
    """Train DistilBERT silently and return metrics dictionary"""

    # Reset GPU memory tracking
    reset_gpu_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    # Initialize model
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        dropout=dropout_rate,
        attention_dropout=dropout_rate
    ).to(device)

    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # Training
    training_start = time.time()

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

    training_time = time.time() - training_start

    # Evaluate on validation set
    val_metrics = evaluate_model(model, val_loader, desc="")

    # GPU memory
    gpu_memory = get_gpu_memory_usage()

    # Clean up
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "training_time": training_time,
        "inference_time_per_batch": val_metrics["inference_time_per_batch"],
        "gpu_memory_mb": gpu_memory
    }

# Quick test
print("Testing fast training function...")
test_result = train_model_fast(epochs=1, lr=2e-5, weight_decay=0.01, warmup_ratio=0.1, dropout_rate=0.1)
print(f"Test run - Loss: {test_result['val_loss']:.4f}, Acc: {test_result['val_accuracy']:.4f}, Time: {test_result['training_time']:.2f}s")
print("✓ Fast training function ready!")

In [ ]:
# Cell 12: Particle Swarm Optimization for Hyperparameter Tuning
import pyswarms as ps

print("=" * 60)
print("PARTICLE SWARM OPTIMIZATION (PSO)")
print("=" * 60)

# PSO Settings
n_particles = 10
n_iterations = 2  # 10 particles × 2 iterations = 20 evaluations
options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}

print(f"Settings: {n_particles} particles × {n_iterations} iterations = {n_particles * n_iterations} evaluations")
print(f"Options: c1={options['c1']}, c2={options['c2']}, w={options['w']}")

# Storage for all evaluated configurations
pso_all_results = []

# Objective function for PSO
def pso_objective(particles):
    """PSO objective: minimize validation loss (maximize negative f1)"""
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)

    for i in range(n_particles):
        lr = particles[i, 0]
        epochs = int(particles[i, 1])
        wd = particles[i, 2]
        warmup = particles[i, 3]
        dropout = particles[i, 4]

        print(f"  PSO eval {i+1}/{n_particles}: lr={lr:.2e}, epochs={epochs}, wd={wd:.2e}, warmup={warmup:.3f}, dropout={dropout:.3f}")

        try:
            result = train_model_fast(
                epochs=epochs, lr=lr, weight_decay=wd,
                warmup_ratio=warmup, dropout_rate=dropout
            )
            # Objective: minimize validation loss (you can use -f1 for maximizing f1)
            scores[i] = result["val_loss"]

            # Store all results
            pso_all_results.append({
                "lr": lr, "epochs": epochs, "weight_decay": wd,
                "warmup_ratio": warmup, "dropout_rate": dropout,
                **result
            })

        except Exception as e:
            print(f"  ✗ Error: {e}")
            scores[i] = 999.0

    return scores

# Define bounds
bounds = (np.array(lb), np.array(ub))

# Run PSO
print("\nRunning PSO optimization...")
pso_start = time.time()

optimizer = ps.single.GlobalBestPSO(
    n_particles=n_particles,
    dimensions=5,
    options=options,
    bounds=bounds
)

best_cost, best_pos = optimizer.optimize(pso_objective, iters=n_iterations)
pso_time = time.time() - pso_start

# Best results
pso_best = {
    "algorithm": "PSO",
    "learning_rate": best_pos[0],
    "epochs": int(best_pos[1]),
    "weight_decay": best_pos[2],
    "warmup_ratio": best_pos[3],
    "dropout_rate": best_pos[4],
    "best_val_loss": best_cost,
    "optimization_time": pso_time
}

print(f"\n{'='*60}")
print("PSO BEST RESULTS:")
print(f"{'='*60}")
for k, v in pso_best.items():
    if isinstance(v, float) and k != "algorithm":
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")
print(f"{'='*60}")

# Find best by F1-score
best_by_f1 = max(pso_all_results, key=lambda x: x['val_f1'])
print(f"\nBest by F1-Score: {best_by_f1['val_f1']:.4f} (Loss: {best_by_f1['val_loss']:.4f})")

In [ ]:
# Cell 13: Genetic Algorithm for Hyperparameter Tuning
from deap import base, creator, tools, algorithms

print("=" * 60)
print("GENETIC ALGORITHM (GA)")
print("=" * 60)

# Clear previous DEAP setup if exists
if 'FitnessMin' in creator.__dict__:
    del creator.FitnessMin
if 'Individual' in creator.__dict__:
    del creator.Individual

# GA Settings
POP_SIZE = 10
N_GENERATIONS = 2  # 10 population × 2 generations = 20 evaluations
CX_PROB = 0.7
MUT_PROB = 0.3

print(f"Settings: Population={POP_SIZE} × Generations={N_GENERATIONS} = {POP_SIZE * N_GENERATIONS} evaluations")
print(f"Crossover prob={CX_PROB}, Mutation prob={MUT_PROB}")

# Storage
ga_all_results = []

# Create fitness and individual
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # Minimize loss
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Hyperparameter generators
toolbox.register("attr_lr", lambda: 10 ** np.random.uniform(np.log10(lb[0]), np.log10(ub[0])))
toolbox.register("attr_epochs", lambda: np.random.randint(lb[1], ub[1] + 1))
toolbox.register("attr_wd", lambda: 10 ** np.random.uniform(np.log10(lb[2]), np.log10(ub[2])))
toolbox.register("attr_warmup", lambda: np.random.uniform(lb[3], ub[3]))
toolbox.register("attr_dropout", lambda: np.random.uniform(lb[4], ub[4]))

# Individual and population
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_lr, toolbox.attr_epochs, toolbox.attr_wd,
                  toolbox.attr_warmup, toolbox.attr_dropout), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Evaluation function
def ga_evaluate(individual):
    lr, epochs, wd, warmup, dropout = individual
    epochs = int(epochs)

    print(f"  GA eval: lr={lr:.2e}, epochs={epochs}, wd={wd:.2e}, warmup={warmup:.3f}, dropout={dropout:.3f}")

    try:
        result = train_model_fast(
            epochs=epochs, lr=lr, weight_decay=wd,
            warmup_ratio=warmup, dropout_rate=dropout
        )

        ga_all_results.append({
            "lr": lr, "epochs": epochs, "weight_decay": wd,
            "warmup_ratio": warmup, "dropout_rate": dropout,
            **result
        })

        return (result["val_loss"],)
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return (999.0,)

# Crossover (blend crossover)
def ga_crossover(ind1, ind2):
    for i in range(len(ind1)):
        alpha = np.random.random()
        ind1[i], ind2[i] = ind1[i] + alpha * (ind2[i] - ind1[i]), ind2[i] + alpha * (ind1[i] - ind2[i])

    # Fix epochs to int
    ind1[1] = int(round(np.clip(ind1[1], lb[1], ub[1])))
    ind2[1] = int(round(np.clip(ind2[1], lb[1], ub[1])))

    # Clip all values
    for ind in [ind1, ind2]:
        ind[0] = np.clip(ind[0], lb[0], ub[0])
        ind[2] = np.clip(ind[2], lb[2], ub[2])
        ind[3] = np.clip(ind[3], lb[3], ub[3])
        ind[4] = np.clip(ind[4], lb[4], ub[4])

    return ind1, ind2

# Mutation
def ga_mutate(individual):
    idx = np.random.randint(0, 5)
    if idx == 0:
        individual[idx] = 10 ** np.random.uniform(np.log10(lb[0]), np.log10(ub[0]))
    elif idx == 1:
        individual[idx] = np.random.randint(lb[1], ub[1] + 1)
    elif idx == 2:
        individual[idx] = 10 ** np.random.uniform(np.log10(lb[2]), np.log10(ub[2]))
    elif idx == 3:
        individual[idx] = np.random.uniform(lb[3], ub[3])
    elif idx == 4:
        individual[idx] = np.random.uniform(lb[4], ub[4])
    return (individual,)

# Register operators
toolbox.register("evaluate", ga_evaluate)
toolbox.register("mate", ga_crossover)
toolbox.register("mutate", ga_mutate)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run GA
print("\nRunning Genetic Algorithm...")
population = toolbox.population(n=POP_SIZE)
ga_start = time.time()

for gen in range(N_GENERATIONS):
    print(f"\n--- Generation {gen+1}/{N_GENERATIONS} ---")

    # Evaluate
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit

    best_ind = tools.selBest(population, 1)[0]
    print(f"  Best loss: {best_ind.fitness.values[0]:.4f}")

    # Select and breed
    offspring = toolbox.select(population, len(population))
    offspring = list(map(toolbox.clone, offspring))

    # Crossover
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if np.random.random() < CX_PROB:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    # Mutation
    for mutant in offspring:
        if np.random.random() < MUT_PROB:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    population[:] = offspring

ga_time = time.time() - ga_start

# Best results
best_ga_ind = tools.selBest(population, 1)[0]
ga_best = {
    "algorithm": "GA",
    "learning_rate": best_ga_ind[0],
    "epochs": int(best_ga_ind[1]),
    "weight_decay": best_ga_ind[2],
    "warmup_ratio": best_ga_ind[3],
    "dropout_rate": best_ga_ind[4],
    "best_val_loss": best_ga_ind.fitness.values[0],
    "optimization_time": ga_time
}

print(f"\n{'='*60}")
print("GA BEST RESULTS:")
print(f"{'='*60}")
for k, v in ga_best.items():
    if isinstance(v, float) and k != "algorithm":
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")
print(f"{'='*60}")

# Best by F1
best_ga_f1 = max(ga_all_results, key=lambda x: x['val_f1'])
print(f"\nBest by F1-Score: {best_ga_f1['val_f1']:.4f} (Loss: {best_ga_f1['val_loss']:.4f})")